# Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")


## Summarization Middleware 

Automatically summarize conversation history when approach a token limits, preserving recent messages while compressing older context.
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

## message based summarization
agent = create_agent(
    model="ollama:llama3.2",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="ollama:llama3.2",
            trigger=("messages", 10), # the trigger point when the message length == 10 
            keep=("messages", 4) # and after summarization keep the 4 top messages 
        )
    ]
)

In [16]:
## run with thread_id

config={"configurable": {"thread_id": "test-1"}}

In [17]:
questions= [
    "what is 2+2",
    "what is 2-2",
    "what is 0+0",
    "what is 2*3",
    "what is 2*2",
    "what is 2*4",
]

for q in questions:
    response=agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response["messages"])}")

Messages: {'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='e223b13c-d984-42b6-a104-071e9cf56c3d'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-11-04T12:02:52.515259Z', 'done': True, 'done_reason': 'stop', 'total_duration': 372423708, 'load_duration': 6715416, 'prompt_eval_count': 31, 'prompt_eval_duration': 132685000, 'eval_count': 8, 'eval_duration': 217179000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a1a670-ca69-7238-b34e-250165d48bfc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 8, 'total_tokens': 39})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='e223b13c-d984-42b6-a104-071e9cf56c3d'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-1

## Token Size

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
1. Grand Hotel - 5 star, 3000/night, spa, pool, gym
2. City Inn - 4 star, 2000/night, business center
3. Budget Stay - 2 star, 1400/night, free wifi
"""

agent = create_agent(
    model="ollama:llama3.2",
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    system_prompt="You are a helpful AI Assistance.",
    middleware=[
        SummarizationMiddleware(
            model="ollama:llama3.2",
            trigger=("tokens", 500),
            keep=("tokens", 200)
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages) // 4
    return total_chars


In [19]:
cities=["delhi", "mumbai", "london", "dubai"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]} ,
        config=config
    )
    
    tokens=count_tokens(response["messages"])
    print(f"{city}: {tokens} tokens, {len(response["messages"])} messages")
    print(f"{(response["messages"])}")

delhi: 297 tokens, 4 messages
[HumanMessage(content='Find hotels in delhi', additional_kwargs={}, response_metadata={}, id='9135543a-596b-484e-ba36-c7ef4cfbe899'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-11-05T10:45:13.161966Z', 'done': True, 'done_reason': 'stop', 'total_duration': 676706000, 'load_duration': 43117125, 'prompt_eval_count': 165, 'prompt_eval_duration': 106508000, 'eval_count': 16, 'eval_duration': 379647000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a1ab50-0c94-7165-bf0d-012fa406b536-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'delhi'}, 'id': 'afe6364d-01b2-4457-9e4e-998ed9bff9b6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 165, 'output_tokens': 16, 'total_tokens': 181}), ToolMessage(content='Hotels in delhi:\n1. Grand Hotel - 5 star, 3000/night, spa, pool, gym\n2. City Inn - 4 star, 2000/night, business center\n

## Fraction

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
1. Grand Hotel - 5 star, 3000/night, spa, pool, gym
2. City Inn - 4 star, 2000/night, business center
3. Budget Stay - 2 star, 1400/night, free wifi
"""

llm = init_chat_model(
    "ollama:llama3.2",
    profile={
        "max_input_tokens": 128000
    }
)

agent = create_agent(
    model="ollama:llama3.2",
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    system_prompt="You are a helpful AI Assistance.",
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.005),
            keep=("fraction", 0.002)
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages) // 4
    return total_chars

cities = ["Delhi", "Roorkee", "Noida", "Mumbai", "Paris"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [HumanMessage(content=f"Hotels in {city}")]},
            config=config        
    )
    tokens = count_tokens(response["messages"])
    fraction=tokens/128000
    print(f"{city}: ~{tokens} tokens ({fraction: .4%}), {len(response["messages"])} msgs")
    print(response["messages"])


Delhi: ~312 tokens ( 0.2437%), 4 msgs
[HumanMessage(content='Hotels in Delhi', additional_kwargs={}, response_metadata={}, id='672e344f-edc9-45b3-ae28-2ae991d9bcae'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-11-05T10:58:19.067407Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16059175500, 'load_duration': 14644994792, 'prompt_eval_count': 164, 'prompt_eval_duration': 729935000, 'eval_count': 19, 'eval_duration': 583901000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a1ab5b-ce5b-72ee-95db-7c0317700d58-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Delhi'}, 'id': '3e55fdfd-796d-43ad-b0b6-983bb0516d71', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 164, 'output_tokens': 19, 'total_tokens': 183}), ToolMessage(content='Hotels in Delhi:\n1. Grand Hotel - 5 star, 3000/night, spa, pool, gym\n2. City Inn - 4 star, 2000/night, business 

# Human in the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human in the loop is useful for the following.
- High-stakes operations required human approval (eg. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.